In [70]:
import os
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from collections import Counter
from datasets import Dataset, DatasetDict
import evaluate

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from seqeval.metrics import classification_report
from torch.nn import CrossEntropyLoss

os.environ["WANDB_DISABLED"] = "true"

In [71]:
DATA_DIR = pathlib.Path("/kaggle/input/datasets/elshadaydesalegn/amharic-ner-data")
TRAIN_FILE = DATA_DIR / "train.txt"
DEV_FILE   = DATA_DIR / "dev.txt"
TEST_FILE  = DATA_DIR / "test.txt"

MODEL_CHECKPOINT = "xlm-roberta-base"
OUTPUT_DIR = "./ner_amharic"

NUM_TRAIN_EPOCHS = 10
PER_DEVICE_TRAIN_BATCH_SIZE = 16
PER_DEVICE_EVAL_BATCH_SIZE  = 16
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.05
LOG_DIR = "./logs"

In [72]:
def read_conll(path):
    sentences, labels = [], []
    tokens, tags = [], []

    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                if tokens:
                    sentences.append(tokens)
                    labels.append(tags)
                    tokens, tags = [], []
            else:
                parts = line.split()
                tokens.append(" ".join(parts[:-1]))
                tags.append(parts[-1])

        if tokens:
            sentences.append(tokens)
            labels.append(tags)

    return sentences, labels


In [73]:
train_tokens, train_tags = read_conll(TRAIN_FILE)
dev_tokens, dev_tags     = read_conll(DEV_FILE)
test_tokens, test_tags   = read_conll(TEST_FILE)

label_list = sorted(set(
    t for s in train_tags + dev_tags + test_tags for t in s
))
label_to_id = {l: i for i, l in enumerate(label_list)}

print("\nLabel → ID mapping:")
for l, i in label_to_id.items():
    print(f"{i:2d} -> {l}")

def encode(tags):
    return [[label_to_id[t] for t in sent] for sent in tags]

dataset = DatasetDict({
    "train": Dataset.from_dict({"tokens": train_tokens, "ner_tags": encode(train_tags)}),
    "validation": Dataset.from_dict({"tokens": dev_tokens, "ner_tags": encode(dev_tags)}),
    "test": Dataset.from_dict({"tokens": test_tokens, "ner_tags": encode(test_tags)}),
})


Label → ID mapping:
 0 -> B-DATE
 1 -> B-LOC
 2 -> B-ORG
 3 -> B-PER
 4 -> I-DATE
 5 -> I-LOC
 6 -> I-ORG
 7 -> I-PER
 8 -> O


In [74]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def tokenize_and_align(examples):
    tok = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        truncation=True
    )

    labels = []
    for i, labs in enumerate(examples["ner_tags"]):
        word_ids = tok.word_ids(batch_index=i)
        prev = None
        aligned = []

        for w in word_ids:
            if w is None:
                aligned.append(-100)
            elif w != prev:
                aligned.append(labs[w])
            else:
                aligned.append(-100)
            prev = w

        labels.append(aligned)

    tok["labels"] = labels
    return tok

tokenized = dataset.map(tokenize_and_align, batched=True)


Map:   0%|          | 0/1750 [00:00<?, ? examples/s]

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [75]:
from collections import Counter
import torch

counter = Counter()
for ex in tokenized["train"]:
    for l in ex["labels"]:
        if l != -100:
            counter[l] += 1

total = sum(counter.values())
num_labels = len(label_list)

class_weights = torch.zeros(num_labels)
for i in range(num_labels):
    class_weights[i] = total / (counter[i] + 1e-6)

class_weights = class_weights / class_weights.mean()

if "O" in label_to_id:
    class_weights[label_to_id["O"]] *= 0.3

class_weights = class_weights.cuda()


In [76]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(label_list)
)

model.config.id2label = {i: l for i, l in enumerate(label_list)}
model.config.label2id = label_to_id


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.weight           | MISSING    | 
classifier.bias             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [77]:
import torch
from transformers import Trainer
from torch.nn import CrossEntropyLoss

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        loss_fct = CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        loss = loss_fct(
            logits.view(-1, logits.size(-1)),   # was: model.config.num_labels
            labels.view(-1)
        )

        return (loss, outputs) if return_outputs else loss

In [78]:
def compute_metrics(p):
    preds, labels = p
    preds = np.argmax(preds, axis=2)

    true_preds, true_labels = [], []
    for pr, lb in zip(preds, labels):
        tp, tl = [], []
        for p_i, l_i in zip(pr, lb):
            if l_i != -100:
                tp.append(label_list[p_i])
                tl.append(label_list[l_i])
        true_preds.append(tp)
        true_labels.append(tl)

    res = metric.compute(predictions=true_preds, references=true_labels)
    return {
        "f1": res["overall_f1"],
        "accuracy": res["overall_accuracy"],
    }


In [79]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT, num_labels=len(label_list)
)
model.config.id2label = {i: l for i, l in enumerate(label_list)}
model.config.label2id = label_to_id

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,          # new
    save_only_model=True,        # new
    logging_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
    push_to_hub=False,
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.weight           | MISSING    | 
classifier.bias             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [80]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)
trainer.class_weights = class_weights.to(model.device)

trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

Epoch,Training Loss,Validation Loss,F1,Accuracy
1,1.606222,0.818220,0.112830,0.130435
2,0.676567,0.441334,0.202107,0.471059
3,0.412961,0.364096,0.297661,0.671379
4,0.318983,0.348318,0.340524,0.718592
5,0.245315,0.365606,0.315871,0.701254
6,0.177810,0.315519,0.383016,0.771939
7,0.140610,0.333314,0.417189,0.790344
8,0.110264,0.320411,0.441093,0.811416
9,0.105333,0.328377,0.451489,0.820485
10,0.091259,0.330707,0.457644,0.825287


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (3 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./ner_amharic/tokenizer_config.json', './ner_amharic/tokenizer.json')

In [81]:
import evaluate
metric = evaluate.load("seqeval")


In [82]:
trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

Epoch,Training Loss,Validation Loss,F1,Accuracy
1,0.121345,0.479747,0.421435,0.813817
2,0.114485,0.531030,0.489752,0.851160
3,0.085404,0.472159,0.534054,0.866631
4,0.090732,0.468901,0.553473,0.869832
5,0.058697,0.437889,0.595647,0.890638
6,0.046969,0.425676,0.603027,0.892505
7,0.041934,0.442476,0.622485,0.900507
8,0.032676,0.471108,0.641371,0.906909
9,0.026241,0.502839,0.670000,0.918645
10,0.026388,0.512011,0.669182,0.919445


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (3 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./ner_amharic/tokenizer_config.json', './ner_amharic/tokenizer.json')

In [83]:
import os, zipfile
from google.colab import files

OUTPUT_DIR = "./ner_amharic"
ZIP_NAME = "ner_amharic_model.zip"

# Pack only the final model files and skip the bulky checkpoint-XXXX folders
keep = [f for f in os.listdir(OUTPUT_DIR)
        if os.path.isfile(os.path.join(OUTPUT_DIR, f)) and f != "training_args.bin"]

with zipfile.ZipFile(ZIP_NAME, "w", zipfile.ZIP_DEFLATED) as z:
    for f in keep:
        z.write(os.path.join(OUTPUT_DIR, f), arcname=f)

print(f"Packed {len(keep)} files:", keep)
print(f"Size: {os.path.getsize(ZIP_NAME)/1e6:.0f} MB")
files.download(ZIP_NAME)

Packed 4 files: ['model.safetensors', 'tokenizer.json', 'tokenizer_config.json', 'config.json']
Size: 828 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [84]:
from seqeval.metrics import classification_report

# ---------------------------
# Evaluate on TRAIN set
# ---------------------------
train_metrics = trainer.evaluate(tokenized["train"])
print("\nTrain metrics:")
for k, v in train_metrics.items():
    print(f"{k}: {v:.4f}")


Training Loss,Validation Loss,Epoch,F1,Accuracy
0.026388,0.009027,10,0.817473,0.960610



Train metrics:
eval_loss: 0.0090
eval_f1: 0.8175
eval_accuracy: 0.9606


In [85]:
from sklearn.metrics import classification_report

predictions, labels, _ = trainer.predict(tokenized["test"])
predictions = np.argmax(predictions, axis=2)

true_preds, true_labels = [], []
for pr, lb in zip(predictions, labels):
    for p_i, l_i in zip(pr, lb):
        if l_i != -100:
            true_preds.append(label_list[p_i])
            true_labels.append(label_list[l_i])

print("\nClassification Report:")
print(classification_report(true_labels, true_preds))



y_true, y_pred = [], []
for p, l in zip(preds, labels):
    for pi, li in zip(p, l):
        if li != -100:
            y_true.append(li)
            y_pred.append(pi)

cm = confusion_matrix(y_true, y_pred)


Classification Report:
              precision    recall  f1-score   support

      B-DATE       0.63      0.89      0.73       106
       B-LOC       0.64      0.87      0.74       227
       B-ORG       0.43      0.73      0.54        83
       B-PER       0.71      0.92      0.80       142
      I-DATE       0.76      0.89      0.82       163
       I-LOC       0.47      0.95      0.62        74
       I-ORG       0.48      0.83      0.61       114
       I-PER       0.79      0.92      0.85       109
           O       0.99      0.93      0.96      6431

    accuracy                           0.92      7449
   macro avg       0.66      0.88      0.74      7449
weighted avg       0.94      0.92      0.93      7449



NameError: name 'preds' is not defined

In [ ]:
plt.figure(figsize=(12, 12))
disp = ConfusionMatrixDisplay(cm, display_labels=label_list)
disp.plot(cmap="Blues", xticks_rotation=90)
plt.title("Token-Level Confusion Matrix (Test Set)")
plt.tight_layout()
plt.show()

In [ ]:
!pip install --upgrade transformers datasets evaluate seqeval


In [ ]:
!pip install imbalanced-learn


In [ ]:
!rm -rf ./ner_amharic ./logs
!df -h /content | tail -1